In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from glob import glob
from pathlib import Path
from autopath.ap_PLIP import ProteinLigandAnalyzer
import os
import numpy as np

In [ ]:
EXPERIMENT = None  # e.g. 'mmgbsa_GBn2'; None = all experiments
allsystems = sorted(glob('*/mmgbsa_*/FINAL_RESULTS_mmpbsa.dat'))
print(f'Found {len(allsystems)} results.')
alldfs = []
for fname in allsystems:
    p = Path(fname)
    df_diff = ProteinLigandAnalyzer.parse_mmpbsa_differences_table(fname)
    df_diff['system'] = p.parent.parent.name
    df_diff['experiment'] = p.parent.name
    alldfs.append(df_diff)

df_all = pd.concat(alldfs, ignore_index=True)
df_all = df_all[~df_all['Component'].isin(['ENPOLAR', 'EDISPER', 'DELTA G gas', 'DELTA G solv'])]
if EXPERIMENT is not None:
    df_all = df_all[df_all['experiment'] == EXPERIMENT]
os.makedirs('plots_mmgbsa', exist_ok=True)
df_all.head()

### Check components contribution by molecule

In [ ]:
experiments = df_all['experiment'].unique()
fig, axes = plt.subplots(len(experiments), 1, figsize=(8, 4 * len(experiments)), squeeze=False)
for ax, exp in zip(axes[:, 0], experiments):
    d = df_all[df_all['experiment'] == exp]
    order = d.groupby('system')['Average'].sum().sort_values().index
    sns.barplot(data=d, x='system', y='Average', hue='Component', order=order, ax=ax)
    ax.set_title(exp); ax.set_ylabel('Energy (kcal/mol)'); ax.set_xlabel('System')
    ax.legend(title='Component', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('plots_mmgbsa/mmgbsa_percomponent_barplot.png', dpi=300, bbox_inches='tight')

### Check Total Energy for all compounds

In [ ]:
allsystems_total = df_all[df_all['Component'] == 'DELTA TOTAL'].copy()
plt.figure(figsize=(8, 4))
order = allsystems_total.groupby('system')['Average'].mean().sort_values().index
sns.barplot(data=allsystems_total, x='system', y='Average', hue='experiment', order=order)
plt.ylabel('Energy (kcal/mol)'); plt.xlabel('System')
plt.xticks(rotation=45, ha='right')
plt.savefig('plots_mmgbsa/mmgbsa_totalEnergy_barplot.png', dpi=300, bbox_inches='tight')